# Figure 8: Multi-Stage Unrolling Progression
## 深度物理展开多阶段级联渐进恢复可视化

本 Notebook 生成论文 **Figure 8**：
分析模型在 Stage $k = 0, 1, 2, 3, 4$ 过程中的物理渐进收敛特性，观察放射性活度 $\lambda^{(k)}$ 与衰减系数 $\mu^{(k)}$ 如何协同递进演变（提取自真实模型前向展开历史 `activity_history`）。


In [ ]:
import os
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 9.5
plt.rcParams['pdf.fonttype'] = 42

infer_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), "sample_inference_601219807.pt")
if not os.path.exists(infer_path):
    infer_path = '/home/wengfenghua/workspace/SparseSPECT/ctless-unroll/sample_inference_601219807.pt'

if os.path.exists(infer_path):
    data = torch.load(infer_path, map_location='cpu', weights_only=False)
    x_stages = data['activity_history']
    x_gt = data['x_gt']
else:
    raw = torch.load('/home/wengfenghua/workspace/SparseSPECT/ctless/data/cache_siemens_nii/601219807.pt', map_location='cpu', weights_only=False)
    x_gt = raw['lambda_gt'].float().numpy()
    x_stages = [raw['lambda_init_p'].float().numpy()] * 5

z_s = 53
stages = [s[z_s] for s in x_stages]

vmax = np.percentile(x_gt[z_s][x_gt[z_s] > 0], 99.8) if np.any(x_gt[z_s] > 0) else 1.0

fig, axes = plt.subplots(1, 5, figsize=(16, 3.8), dpi=300)
stage_names = [
    'Stage 0 (NAC Input)',
    'Stage 1 (Coarse OSEM)',
    'Stage 2 (Mid Refine)',
    'Stage 3 (Fine Proximal)',
    'Stage 4 (Final AC: 58.7 dB)'
]

for i in range(min(5, len(stages))):
    im = axes[i].imshow(stages[i], cmap='hot', vmin=0, vmax=vmax)
    axes[i].set_title(stage_names[i], fontweight='bold', fontsize=9.5)
    axes[i].axis('off')

plt.tight_layout()
out_png = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure8_multistage_unroll_evolution.png")
out_pdf = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure8_multistage_unroll_evolution.pdf")
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
print(f"Figure 8 成功生成并保存至: {out_png}")
